# Fidelity and trace distance
**Quantum Computing · Week 6 (Noise and real hardware) · Day 3**

Lesson page: https://sara-aminpour.com/tutorials/qc/w6d3.html

In [ ]:
# On Google Colab, this installs qiskit (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2

How close are two quantum states? **Fidelity** (1 = the same) and **trace distance** (0 = the same)
answer that, and the trace distance says exactly how well one measurement can tell them apart.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, DensityMatrix, state_fidelity, Pauli
from qiskit.primitives import StatevectorSampler

def trace_distance(rho, sigma):
    """D = (1/2) * sum of |eigenvalues of rho - sigma|."""
    return 0.5 * np.abs(np.linalg.eigvalsh(DensityMatrix(rho).data - DensityMatrix(sigma).data)).sum()

def bloch(rho):
    return np.array([np.real(DensityMatrix(rho).expectation_value(Pauli(p))) for p in "XYZ"])

zero, one, plus = (DensityMatrix.from_label(s) for s in "01+")
half = DensityMatrix(np.eye(2) / 2)
for name, (a, b) in {"|0> vs |0>": (zero, zero), "|0> vs |+>": (zero, plus),
                     "|0> vs |1>": (zero, one), "|0> vs I/2": (zero, half)}.items():
    F, D = state_fidelity(a, b), trace_distance(a, b)
    print(f"{name}: F = {F:.3f}, D = {D:.3f}, half the distance between the Bloch tips = {0.5 * np.linalg.norm(bloch(a) - bloch(b)):.3f}")

For one qubit, the trace distance is half the distance between the arrows' tips. For two pure states,
$D = \sqrt{1 - F}$: here $\sqrt{1 - 0.5} = 0.707$ for $|0\rangle$ versus $|+\rangle$.

## The best possible guess: the Helstrom bound
Someone sends $\rho$ or $\sigma$, each with probability ½, and you get **one copy**. The best any
measurement can do is $P_\text{correct} = \frac12(1 + D)$. The optimal measurement separates the positive
and negative parts of $\rho - \sigma$; for a qubit, it measures along the axis $\vec r_\rho - \vec r_\sigma$.

In [ ]:
D = trace_distance(zero, plus)
print(f"Helstrom bound for |0> vs |+>: {(1 + D) / 2:.4f}")

def guess_rate(prepare_A, prepare_B, rotation, shots=20000):
    """Rotate, measure in the 0/1 basis, guess A on 0 and B on 1. Returns the share of right guesses."""
    sampler = StatevectorSampler(seed=7)
    right = 0
    for prepare, correct in [(prepare_A, "0"), (prepare_B, "1")]:
        qc = prepare.copy()
        if rotation:
            qc.ry(rotation, 0)
        qc.measure_all()
        counts = sampler.run([qc], shots=shots).result()[0].data.meas.get_counts()
        right += counts.get(correct, 0)
    return right / (2 * shots)

prep_zero = QuantumCircuit(1)
prep_plus = QuantumCircuit(1)
prep_plus.h(0)
print(f"plain 0/1 measurement      : {guess_rate(prep_zero, prep_plus, 0):.3f}")
print(f"along the axis A - B (RY(pi/4) first): {guess_rate(prep_zero, prep_plus, np.pi / 4):.3f}")

Measuring in the plain 0/1 basis wins 75% of the time: $|0\rangle$ is always recognized, $|+\rangle$
only half the time. Turning the measurement to the axis halfway between the two states reaches the
bound, 85.4%, and no measurement can do better. Non-orthogonal states can never be told apart perfectly:
that's the principle behind quantum key distribution.

## Your turn: noise blurs the difference
Depolarizing noise of strength $p$ shrinks every Bloch vector by $1 - p$. Apply it to both $|0\rangle$
and $|+\rangle$, for $p = 0, 0.2, 0.5, 0.8$, and compute the trace distance and the best guess rate.

In [ ]:
def depolarize(rho, p):
    return DensityMatrix((1 - p) * DensityMatrix(rho).data + p * np.eye(2) / 2)

for p in [0.0, 0.2, 0.5, 0.8]:
    D = 0.0          # TODO: trace_distance of the two noisy states
    print(p, D, (1 + D) / 2)

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
def depolarize(rho, p):
    return DensityMatrix((1 - p) * DensityMatrix(rho).data + p * np.eye(2) / 2)

for p in [0.0, 0.2, 0.5, 0.8]:
    D = trace_distance(depolarize(zero, p), depolarize(plus, p))
    print(f"p = {p:.1f}: trace distance {D:.3f}, best guess rate {(1 + D) / 2:.3f}")
print("Noise shrinks the distance by exactly 1 - p: noisy states look more alike, and every measurement")
print("(including a quantum classifier's) has a harder job.")